# vLLM BaseDemand Emergency Evaluation

This notebook runs `agent/base_demand_emergency.py`, a BaseDemand-only emergency version. External trajectory tool calls are restricted to `search` and `get_document`.

## 0. Environment

Run this after the vLLM OpenAI-compatible service is already started on the cloud machine.

In [ ]:
!python --version
!pip install -q openai pandas pyarrow tqdm

## 1. Paths And vLLM Config

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
MODEL_NAME = 'qwen_auto'
API_KEY = 'dummy'

hard50_path = str(project_root / 'browsecomp_plus_hard50.jsonl')
bm25_index_path = str(project_root / 'indexes' / 'browsecomp_plus_bm25.sqlite')
submission_path = str(project_root / 'runs' / 'base_demand_emergency_submission.jsonl')
eval_output_path = str(project_root / 'runs' / 'base_demand_emergency_eval_results.jsonl')

print('project_root:', project_root)
print('dataset:', hard50_path)
print('index:', bm25_index_path)
print('submission:', submission_path)
print('eval:', eval_output_path)

## 2. Build BM25 Index If Needed

If the index already exists, skip this cell.

In [ ]:
!python -m agent.build_bm25_index --corpus-path ./browsecomp-plus-corpus --index-path ./indexes/browsecomp_plus_bm25.sqlite --overwrite

## 3. Initialize Emergency BaseDemand Agent

In [ ]:
import importlib
from agent.vllm_client import VLLMClient
from agent.tools import build_searcher, get_agent_tool_specs_and_registry
from agent.dataset_utils import load_jsonl
import agent.base_demand_emergency as base_demand_emergency

base_demand_emergency = importlib.reload(base_demand_emergency)
run_base_demand_agent = base_demand_emergency.run_base_demand_agent
generate_submission = base_demand_emergency.generate_submission

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=bm25_index_path)
tool_specs, tool_registry = get_agent_tool_specs_and_registry(searcher=searcher, k=5, snippet_max_chars=1200)
print('search_type:', searcher.search_type)

## 4. Parameters

In [ ]:
TOP_K = 5
MAX_ROUNDS = 7
DECISION_MAX_TOKENS = 384
ANSWER_MAX_TOKENS = 512
SEARCH_SNIPPET_MAX_CHARS = 1200
TOOL_CONTENT_MAX_CHARS = 4000
LIMIT = 50

## 5. One-Query Smoke Test

In [ ]:
rows = load_jsonl(hard50_path, limit=1)
demo_row = rows[0]

demo = run_base_demand_agent(
    question=demo_row['query'],
    client=client,
    model_name=MODEL_NAME,
    tool_registry=tool_registry,
    max_rounds=MAX_ROUNDS,
    decision_max_tokens=DECISION_MAX_TOKENS,
    answer_max_tokens=ANSWER_MAX_TOKENS,
    tool_content_max_chars=TOOL_CONTENT_MAX_CHARS,
)

print('query_id:', demo_row['query_id'])
print('gold_answer:', demo_row['answer'])
print('predicted_answer:', demo['predicted_answer'])
print('messages:', len(demo['messages']))
print('tool calls:')
for msg in demo['messages']:
    for call in msg.get('tool_calls', []) if isinstance(msg, dict) else []:
        print('-', call.get('function', {}).get('name'))

## 6. Generate Submission

In [ ]:
rows = load_jsonl(hard50_path, limit=LIMIT)

records = generate_submission(
    dataset_rows=rows,
    index_path=bm25_index_path,
    base_url=VLLM_BASE_URL,
    model_name=MODEL_NAME,
    output_path=submission_path,
    api_key=API_KEY,
    top_k=TOP_K,
    search_snippet_max_chars=SEARCH_SNIPPET_MAX_CHARS,
    tool_content_max_chars=TOOL_CONTENT_MAX_CHARS,
    max_rounds=MAX_ROUNDS,
    decision_max_tokens=DECISION_MAX_TOKENS,
    answer_max_tokens=ANSWER_MAX_TOKENS,
)

print('saved:', submission_path)
print('num_records:', len(records))
print('first predicted:', records[0]['predicted_answer'])

## 7. Evaluate

In [ ]:
from agent.eval import run_evaluation

summary, details = run_evaluation(
    submission_path=submission_path,
    dataset_path=hard50_path,
    model_name=MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key=API_KEY,
    output_path=eval_output_path,
    temperature=0.0,
    max_tokens=256,
    verbose=True,
)

print('summary:', summary)

## 8. Constraint Check

In [ ]:
allowed_tools = {'search', 'get_document'}
used_tools = set()
for record in records:
    for msg in record.get('messages', []):
        for call in msg.get('tool_calls', []) if isinstance(msg, dict) else []:
            used_tools.add(call.get('function', {}).get('name', ''))

print('used_tools:', sorted(used_tools))
print('base_tools_only:', used_tools <= allowed_tools)
print('submission_path:', submission_path)
print('eval_output_path:', eval_output_path)